# BreastDCEDL Explore Dicom data




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
%matplotlib inline
import sys
from pathlib import Path

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
# cb - cotrast_brightness
def show_n_images(imgs, cmap='gray', titles = None, enlarge = 4, mtitle=None,
                  cut = 0, axis_off = True, fontsize=15, cb = 0):

    plt.set_cmap(cmap);

    n = len(imgs);
    gs1 = gridspec.GridSpec(1, n);

    fig1 = plt.figure(figsize=(4*len(imgs),8));
    for i in range(n):

        ax1 = fig1.add_subplot(gs1[i]);
        if (cb):
            if len(np.unique(imgs[i])<=5):
                 img = imgs[i]
            else:

                img = cont_br(imgs[i])
        else:
            img = imgs[i]
        if cut:
            ax1.imshow(img[50:290, 75:450] , interpolation='none', origin='lower');
        else:

            ax1.imshow(img, interpolation='none');
        if (titles is not None):
            ax1.set_title(titles[i], fontsize=fontsize);  #, fontweight="bold");
        if (axis_off):
            plt.axis('off')
    if mtitle:
        plt.title(mtitle)
    plt.tight_layout()
    plt.show();

In [ ]:
!ls

In [ ]:
!ls ISPY2

In [ ]:
!ls ISPY2/TCIA_metadata

In [ ]:

from PIL import Image
import requests
import argparse
import datetime
import json
import numpy as np
import os
import time
from pathlib import Path


In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

In [ ]:
import warnings
warnings.filterwarnings('ignore', '.*do not.*', )
warnings.warn('Do not show this message')

import os
from __future__ import print_function

import glob
from itertools import chain
import os
import random
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
from glob import glob
#from skimage import io
from sklearn.utils import shuffle

#from nipype.interfaces.ants import N4BiasFieldCorrection
import sys
import os
import ast

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Dicom utilies

In [ ]:
import pydicom

# return list of numpy array of voxels
def read_dcm_series(fname, bpath=''):

    fpathf = join_path(bpath,fname)

    ll = os.listdir(fpathf)
    pe=[]
    for p in ll:
            d = pydicom.dcmread(join_path(fpathf,p))
            pe.append(d.pixel_array)
    return pe

# returns list of dcm objectys
def read_dcm_series_dicom(fname, bpath=''):
  
    fpathf = join_path(bpath,fname)

    ll = os.listdir(fpathf)
    pe=[]
    for p in ll:
            d = pydicom.dcmread(join_path(fpathf,p))
            pe.append(d)
    return pe

def get_dicom_val(d, tag=(0x0020, 0x0105)):

    val = d.get(tag).value if tag in d else None
    return val

def get_int_dicom_val(d, tag=(0x0020, 0x0105)):
    
    
    val = int(float(d.get(tag).value)) if tag in d else None
    return val

In [ ]:
def extract_image_positions(dicom):
 

    """
    Extracts Image Position (Patient) values from a multi-frame DICOM file
    and returns them as a pandas DataFrame.

    Parameters:
    dicom: DICOM file.

    Returns:
    pd.DataFrame: DataFrame containing frame number and (X, Y, Z) positions.
    """
    # Load the DICOM file
    dicom_file = dicom

    # Extract all image positions
    image_positions = []

    if "PerFrameFunctionalGroupsSequence" in dicom_file:
        for frame_idx, frame in enumerate(dicom_file.PerFrameFunctionalGroupsSequence):
            if "PlanePositionSequence" in frame:
                position = list(frame.PlanePositionSequence[0].ImagePositionPatient)  # Convert MultiValue to list
                image_positions.append([frame_idx] + position)

    # Create a DataFrame
    df_positions = pd.DataFrame(image_positions, columns=["Frame", "X", "Y", "Z"])

    return df_positions

# Load metadata

In [ ]:
df = pd.read_csv(os.path.join('BreastDCEDL_metadata.csv'))
df.head()

# Load example and visualize

In [ ]:
subject_id = 'ISPY2-550421'

df[df.pid==subject_id].T

# Read DICOM
### Load TCIA metadata

In [ ]:
df_meta = pd.read_csv(os.path.join('ISPY2','TCIA_metadata','metadata.csv'))
df_meta.head()

In [ ]:
def get_visit(x):
    if x[-2:] == 'T0':
        return 'T0'
    elif x[-2:] == 'T1':
        return 'T1'
    elif x[-2:] == 'T2':
        return 'T2'
    elif x[-2:] == 'T3':
        return 'T3'

## Filter Pre Treatment visit

In [ ]:
df_meta['Study Description'] = df_meta['Study Description'].apply(lambda x: get_visit(x))

df_meta['Study Description'].value_counts()

df_meta=df_meta[df_meta['Study Description']=='T0']

# Analiz few examples

In [ ]:
p='ISPY2-208265'
p='ISPY2-550421'
tmp=df_meta[df_meta['Subject ID']==p]
tmp[tmp['Series Description']=='ISPY2 VOLSER uni-lateral cropped original DCE']


In [ ]:
dcm_path = os.path.join("ISPY2","data_samples","dicom_tcia")

In [ ]:
import os
from pathlib import Path

# Extract and clean the raw path
raw_fname = tmp[tmp['Series Description'] == 'ISPY2 VOLSER uni-lateral cropped original DCE']['File Location'].values[0]

# Convert to a proper path object (cross-platform safe)
fname = Path(raw_fname.replace("\\", os.sep).lstrip(".\\/"))

# Build full path
full_path = Path(dcm_path) / fname

# Check and list
if full_path.exists():
    print(list(full_path.iterdir())[:2])
else:
    print("Path does not exist:", full_path)

In [ ]:
!ls ISPY2/data_samples/dicom_tcia/ISPY2/ISPY2-550421/

In [ ]:
!ls ISPY2/data_samples/dicom_tcia/ISPY2/ISPY2-550421/1.3.6.1.4.1.14519.5.2.1.92610516567030038611570123081828666823

In [ ]:
import os
import pydicom
from pathlib import Path

# Normalize fname and build full directory path
clean_fname = str(fname).replace("\\", "/")
dicom_dir = Path(dcm_path) / Path(clean_fname)

# Get the first file in the directory
first_file = sorted(os.listdir(dicom_dir))[0]  # sorted ensures consistent order

# Build full path to the DICOM file
dicom_path = dicom_dir / first_file

# Read the DICOM file
d = pydicom.dcmread(dicom_path)

In [ ]:
d

In [ ]:

tmp=df_meta[df_meta['Subject ID']==p]
tmp[tmp['Series Description']=='ISPY2 VOLSER uni-lateral cropped original DCE']


In [ ]:
df[df.pid==p].T

# Analize few example cases

pds = 'ACRIN-6698-378885'
pdg = 'ISPY2-211408'
pdph = 'ISPY2-550421'

### Read Dynamic Contrast MRI for ISPY2-550421

## Dynamic Contrast MRI Scan Details for ISPY2-550421

- **Total Slices:** 1120  
- **Number of MRI Scans:** 7  
- **Slices per MRI Scan:** 1120 / 7 = **160 slices per scan**  

#### **Acquisition Timing**
- The **first scan** is taken **before contrast injection**.  
- The **remaining 6 scans** are acquired **after injection**, with a time interval of approximately **1.5 minutes** between each.  

#### **Processing Steps**
1. **Group slices by acquisition time** (i.e., pre-contrast and post-contrast phases).  
2. **Order slices spatially** based on their anatomical position to ensure correct reconstruction.  


## 1. **Group slices by acquisition time** (i.e., pre-contrast and post-contrast phases).  

In [ ]:
d

## Dicom Temporal information

```

In [ ]:
pds = 'ACRIN-6698-378885'
pdg = 'ISPY2-211408'
pdph = 'ISPY2-550421'

# Convert data for: ACRIN-6698-378885

### Read Analysis mask of MRI for ACRIN-6698-378885

In [ ]:
tmp=df_meta[df_meta['Subject ID']==p]
tmp[tmp['Series Description'].str.contains('Analysis Mask')]


In [ ]:
fname = tmp[tmp['Series Description'].str.contains('Analysis Mask')]['File Location'].values[0]
fname

In [ ]:
import os
import pydicom
from pathlib import Path

# Normalize fname and build full directory path
clean_fname = str(fname).replace("\\", "/")
dicom_dir = Path(dcm_path) / Path(clean_fname)

# Get the first file in the directory
first_file = sorted(os.listdir(dicom_dir))[0]  # sorted ensures consistent order

# Build full path to the DICOM file
dicom_path = dicom_dir / first_file

# Read the DICOM file
mask_dcm = pydicom.dcmread(dicom_path)

In [ ]:
mask_dcm

In [ ]:
mask_dcm.pixel_array.shape

In [ ]:
df_pos = extract_image_positions(mask_dcm)
df_pos.head()

In [ ]:
df_pos.tail()

In [ ]:
df_pos.describe()

In [ ]:
df.columns

In [ ]:
df[df.pid==p]

7 directories with 160 images each, lets read one dicom to see the information

In [ ]:


print('1-001.dcm Series Number', get_dicom_val(d,(0x0020, 0x0011)))
print('1-001.dcm Acquisition Number', get_dicom_val(d,(0x0020, 0x0012)))
print('1-001.dcm Instance Number', get_dicom_val(d,(0x0020, 0x0013)))
print('1-001.dcm Number of temporal aquisitions:', get_dicom_val(d,(0x0020, 0x0103)))

print('1-001.dcm in_stack_pos', get_dicom_val(d,(0x0020, 0x9057)))
print('1-001.dcm Acq_tm', get_int_dicom_val(d,(0x0008, 0x0032)))
print('1-001.dcm Cont_tm', get_int_dicom_val(d,(0x0008, 0x0033)))
print('1-001.dcm temporal_id', d.get((0x0020, 0x0100)).value if (0x0020, 0x0100) in d else None)
print('1-001.dcm n_tmp_pos_post', d.get((0x0020, 0x0105)).value if (0x0020, 0x0105) in d else None)
print('1-001.dcm n_tmp_pos_total', d.get((0x0117, 0x1030)).value if (0x0117, 0x1030) in d else None)

### Read Analysis mask of MRI for ACRIN-6698-378885

In [ ]:
idx=[75,80,85]
show_n_images([mask_dcm.pixel_array[i] for i in idx],cmap='jet')

In [ ]:
im=mask_dcm.pixel_array[80][70:270,250:350].copy()
im.shape

In [ ]:
im=mask_dcm.pixel_array[80].copy()
im1 = np.where(im==1,1,0)
im2 = np.where(im==17,1,0)
im3 = np.where((im==17 )| (im==1),1,0)
show_n_images([mask_dcm.pixel_array[80],im1,im2,im3])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the gray level to RGB mapping
gray_to_rgb = {
    0: (0, 0, 0),         # Black
    1: (255, 0, 0),       # Red
    2: (0, 255, 0),       # Green
    17: (0, 0, 255),      # Blue
    32: (255, 255, 0),    # Yellow
    33: (255, 0, 255),    # Magenta
    34: (0, 255, 255),    # Cyan
    49: (128, 128, 128)   # Gray
}

# Sample 2D NumPy array (replace with your data)
gray_image = mask_dcm.pixel_array[80].copy()

def map_mask_colors(gray_image):

    # Create an empty RGB image
    rgb_image = np.zeros((gray_image.shape[0], gray_image.shape[1], 3), dtype=np.uint8)

    # Apply the mapping
    for gray_value, rgb_color in gray_to_rgb.items():
        rgb_image[gray_image == gray_value] = rgb_color
    return rgb_image


In [ ]:
im=mask_dcm.pixel_array[80].copy()
im1 = np.where(im==1,1,0)
im2 = np.where(im==0,1,0)
im3 = np.where(im==17,17,0)
im4 = np.where(im==2,33,0)
show_n_images([map_mask_colors(im), map_mask_colors(im1),
               map_mask_colors(im3), map_mask_colors(im4),map_mask_colors(im2)])

In [ ]:
im=mask_dcm.pixel_array[160-62].copy()
im1 = np.where(im==1,1,0)
im2 = np.where(im==0,2,0)
im3 = np.where(im==17,17,0)
im4 = np.where(im==2,33,0)
show_n_images([map_mask_colors(im), map_mask_colors(im1),
               map_mask_colors(im3), map_mask_colors(im4),map_mask_colors(im2)])

In [ ]:
np.unique(mask_dcm.pixel_array.flatten())